In [ ]:
pip install tabpfn-client

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
folder_path = '/content/drive/My Drive/Datasets'
os.makedirs(folder_path, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import seaborn as sns

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from statsmodels.formula.api import ols
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.multitest import multipletests

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

ALPHA = 0.05

print("=" * 70)
print("  SECTION 1 - DATASET ANALYSIS (Python)")
print("=" * 70)

# -----------------------------------------------------------------------------
# 0. LOAD DATA
# -----------------------------------------------------------------------------
df = pd.read_csv(os.path.join(folder_path, "STUDENT06_DATASET_coffee_aroma.csv"))

origin_order   = list(df["Origin"].unique())
roasting_order = ["L", "M", "D"]
df["Origin"]   = pd.Categorical(df["Origin"], categories=origin_order, ordered=False)
df["Roasting"] = pd.Categorical(df["Roasting"], categories=roasting_order, ordered=True)

QUANT_VARS = df.columns[2:].tolist()
Q1, Qp     = QUANT_VARS[0], QUANT_VARS[-1]
Orig1, Orig2 = origin_order[0], origin_order[1]
N_QUANT    = len(QUANT_VARS)

ORIGIN_COLORS = {
    'Brazil'      : '#1f78b4',
    'Colombia'    : '#33a02c',
    'India Cherry': '#e31a1c',
    'Peru'        : '#ff7f00',
    'Uganda'      : '#6a3d9a',
    'Vietnam'     : '#a65628'
}
ROAST_COLORS = {'L': '#a6cee3', 'M': '#fdbf6f', 'D': '#e31a1c'}
ROAST_MARKERS = {'L': 'o', 'M': '^', 'D': 's'}

print(f"\nDataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Quantitative variables ({N_QUANT}): {QUANT_VARS}")
print(f"Q1 (first quant var) = {Q1}  |  Qp (last quant var) = {Qp}")
print(f"Origin1 = {Orig1}  |  Origin2 = {Orig2}")
print(f"\nOrigin counts:\n{df['Origin'].value_counts().to_string()}")
print(f"\nRoasting counts:\n{df['Roasting'].value_counts().to_string()}\n")

# =============================================================================
# 1-A. VARIABLE TYPE IDENTIFICATION
# =============================================================================
print("=" * 70)
print("1-A. VARIABLE TYPES")
print("=" * 70)
print("""
COLUMN         | TYPE                       | SCALE  | REASON
-----------------------------------------------------------------------
Origin         | Categorical NOMINAL         | -      | Country names, no natural order
Roasting       | Categorical ORDINAL         | -      | L < M < D (degree of roasting)
<all 13 vars>  | Quantitative CONTINUOUS     | Ratio  | Concentrations; true zero exists;
               |                             |        | meaningful ratios and differences
""")

# =============================================================================
# 1-B. BOXPLOTS BY ORIGIN AND ROASTING
# =============================================================================
print("1-B. Generating boxplots by Origin and by Roasting...")

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, var in enumerate(QUANT_VARS):
    ax = axes[i]
    data_per_origin = [df[df["Origin"] == o][var].values for o in origin_order]
    bp = ax.boxplot(data_per_origin, label=origin_order, patch_artist=True,
                     showfliers=True, flierprops=dict(markersize=3, alpha=0.5))
    for patch, o in zip(bp['boxes'], origin_order):
        patch.set_facecolor(ORIGIN_COLORS[o])
        patch.set_alpha(0.8)
    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', rotation=40, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)
for j in range(len(QUANT_VARS), len(axes)):
    axes[j].axis("off")
fig.suptitle("Boxplots of Volatile Compound Concentrations by Origin", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("01_boxplots_by_origin.png", dpi=150)
plt.close()

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, var in enumerate(QUANT_VARS):
    ax = axes[i]
    data_per_roast = [df[df["Roasting"] == r][var].values for r in roasting_order]
    bp = ax.boxplot(data_per_roast, label=["Light", "Medium", "Dark"], patch_artist=True,
                     showfliers=True, flierprops=dict(markersize=3, alpha=0.5))
    for patch, r in zip(bp['boxes'], roasting_order):
        patch.set_facecolor(ROAST_COLORS[r])
        patch.set_alpha(0.8)
    ax.set_title(var, fontsize=10, fontweight='bold')
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=7)
for j in range(len(QUANT_VARS), len(axes)):
    axes[j].axis("off")
fig.suptitle("Boxplots of Volatile Compound Concentrations by Roasting Level", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("01_boxplots_by_roasting.png", dpi=150)
plt.close()
print("  -> Saved: 01_boxplots_by_origin.png, 01_boxplots_by_roasting.png\n")

# =============================================================================
# 1-C. COMPARE Q1 AND Qp BETWEEN Origin1 AND Origin2
# =============================================================================
print("=" * 70)
print(f"1-C. PAIRWISE COMPARISON: {Orig1} vs {Orig2}")
print("=" * 70)
print("""
DECISION LOGIC:
  Step 1 -> Shapiro-Wilk test for normality in EACH group (H0: data is normal)
  Step 2 -> If BOTH normal:
              Levene's test for equal variances (H0: var1 = var2)
              p > 0.05 -> Student's t-test (equal_var=True)
              p < 0.05 -> Welch's  t-test (equal_var=False)
            If ANY group non-normal:
              Mann-Whitney U test (non-parametric)
""")

def two_group_test(var, g1, g2, data, tag):
    sub = data[data["Origin"].isin([g1, g2])].copy()
    x1 = sub.loc[sub["Origin"] == g1, var].values
    x2 = sub.loc[sub["Origin"] == g2, var].values

    desc = pd.DataFrame({
        "Group": [g1, g2],
        "n": [len(x1), len(x2)],
        "Mean": [x1.mean(), x2.mean()],
        "SD": [x1.std(ddof=1), x2.std(ddof=1)],
        "Median": [np.median(x1), np.median(x2)]
    }).round(4)
    print(desc.to_string(index=False))

    sw1 = stats.shapiro(x1)
    sw2 = stats.shapiro(x2)
    print(f"Shapiro-Wilk {g1}: W={sw1.statistic:.4f}, p={sw1.pvalue:.5f} "
          f"{'[NON-NORMAL]' if sw1.pvalue < ALPHA else '[Normal]'}")
    print(f"Shapiro-Wilk {g2}: W={sw2.statistic:.4f}, p={sw2.pvalue:.5f} "
          f"{'[NON-NORMAL]' if sw2.pvalue < ALPHA else '[Normal]'}")

    lev = stats.levene(x1, x2, center="median")
    print(f"Levene's test: F={lev.statistic:.4f}, p={lev.pvalue:.5f} "
          f"{'[UNEQUAL var]' if lev.pvalue < ALPHA else '[Equal var]'}")

    both_normal = sw1.pvalue > ALPHA and sw2.pvalue > ALPHA
    equal_var   = lev.pvalue > ALPHA

    if not both_normal:
        stat, p = stats.mannwhitneyu(x1, x2, alternative="two-sided")
        test_name, stat_label = "Mann-Whitney U", "U"
    elif equal_var:
        stat, p = stats.ttest_ind(x1, x2, equal_var=True)
        test_name, stat_label = "Student's t-test (pooled variance)", "t"
    else:
        stat, p = stats.ttest_ind(x1, x2, equal_var=False)
        test_name, stat_label = "Welch's t-test (unequal variance)", "t"

    print(f"\nTest chosen: {test_name}")
    print(f"H0: No difference between {g1} and {g2} for {var}")
    print(f"H1: A significant difference exists")
    print(f"{stat_label} = {stat:.4f}  |  p-value = {p:.6f}  |  alpha = {ALPHA}")
    decision = (f"REJECT H0 -> significant difference between {g1} and {g2}"
                if p < ALPHA else "FAIL to reject H0 -> no significant difference")
    print(f"Decision: {decision}")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.hist(x1, bins=20, alpha=0.45, density=True, color=ORIGIN_COLORS.get(g1, '#2166ac'), label=g1)
    ax.hist(x2, bins=20, alpha=0.45, density=True, color=ORIGIN_COLORS.get(g2, '#d6604d'), label=g2)
    ax.axvline(x1.mean(), color=ORIGIN_COLORS.get(g1, '#2166ac'), linestyle='--', linewidth=1.5)
    ax.axvline(x2.mean(), color=ORIGIN_COLORS.get(g2, '#d6604d'), linestyle='--', linewidth=1.5)
    ax.set_title(f"Distribution of {var}: {g1} vs {g2}\n{test_name} | p = {p:.4f}")
    ax.set_xlabel(var); ax.set_ylabel("Density"); ax.legend()
    plt.tight_layout()
    plt.savefig(f"02_dist_{tag}.png", dpi=150)
    plt.close()
    print(f"  -> Saved: 02_dist_{tag}.png")
    return p

two_group_test(Q1, Orig1, Orig2, df, "Q1_" + Q1)
two_group_test(Qp, Orig1, Orig2, df, "Qp_" + Qp)

# =============================================================================
# 1-D. MULTIPLE COMPARISONS (all vars, Origin1 vs Origin2) + FWER
# =============================================================================
print("\n" + "=" * 70)
print(f"1-D. MULTIPLE COMPARISONS (all {N_QUANT} vars, {Orig1} vs {Orig2}) + FWER")
print("=" * 70)

fwer_uncorrected = 1 - (1 - ALPHA) ** N_QUANT
alpha_bonf = ALPHA / N_QUANT
alpha_sidak = 1 - (1 - ALPHA) ** (1 / N_QUANT)
print(f"""
WHY FWER MATTERS:
  Testing {N_QUANT} hypotheses at alpha={ALPHA} simultaneously:
  FWER = 1 - (1-alpha)^p = {fwer_uncorrected:.4f}  ({fwer_uncorrected*100:.1f}% chance
  of at least one false positive if all H0 are true)

CORRECTIONS:
  Bonferroni : alpha_corrected = alpha/p = {alpha_bonf:.5f}
  Sidak      : alpha_corrected = 1-(1-alpha)^(1/p) = {alpha_sidak:.5f}
""")

sub13 = df[df["Origin"].isin([Orig1, Orig2])]
pvals_13 = []
for v in QUANT_VARS:
    x1 = sub13.loc[sub13["Origin"] == Orig1, v].values
    x2 = sub13.loc[sub13["Origin"] == Orig2, v].values
    sw1, sw2 = stats.shapiro(x1).pvalue, stats.shapiro(x2).pvalue
    if sw1 > ALPHA and sw2 > ALPHA:
        _, p = stats.ttest_ind(x1, x2)
    else:
        _, p = stats.mannwhitneyu(x1, x2, alternative="two-sided")
    pvals_13.append(p)

reject_bonf, p_bonf, _, _ = multipletests(pvals_13, alpha=ALPHA, method="bonferroni")
tbl_13 = pd.DataFrame({
    "Variable": QUANT_VARS,
    "p_uncorrected": np.round(pvals_13, 6),
    "p_Bonferroni": np.round(p_bonf, 6),
    "Sig_uncorrected": np.array(pvals_13) < ALPHA,
    "Sig_Bonferroni": reject_bonf
}).sort_values("p_uncorrected").reset_index(drop=True)

print(tbl_13.to_string(index=False))
tbl_13.to_csv("03_multiple_comparison_13.csv", index=False)
print("  -> Saved: 03_multiple_comparison_13.csv\n")

# =============================================================================
# 1-E. ONE-WAY ANOVA - ORIGIN (all origins, all quant vars)
# =============================================================================
print("=" * 70)
print("1-E. ONE-WAY ANOVA: Effect of ORIGIN")
print("=" * 70)
print("""
MODEL : yij = mu + alpha_i + eps_ij
H0 : all Origin means are equal
H1 : at least one Origin mean differs

PRACTICAL SIGNIFICANCE - ETA-SQUARED (eta2):
  eta2 = SS_between / SS_total
  <0.01 negligible | 0.01-0.06 small | 0.06-0.14 medium | >0.14 large
""")

def eta_squared_oneway(data, var, factor):
    model = ols(f"{var} ~ C({factor})", data=data).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    ss_effect = aov.loc[f"C({factor})", "sum_sq"]
    ss_total  = aov["sum_sq"].sum()
    F = aov.loc[f"C({factor})", "F"]
    p = aov.loc[f"C({factor})", "PR(>F)"]
    return F, p, ss_effect / ss_total

rows = []
for v in QUANT_VARS:
    F, p, eta2 = eta_squared_oneway(df, v, "Origin")
    rows.append({"Variable": v, "F_value": F, "p_value": p, "eta2": eta2})
tbl_anova_origin = pd.DataFrame(rows).sort_values(by=["p_value", "F_value"], ascending=[True, False]).reset_index(drop=True)

def effect_label(e):
    if e < 0.01: return "negligible"
    if e < 0.06: return "small"
    if e < 0.14: return "medium"
    return "large"
tbl_anova_origin["effect_size"] = tbl_anova_origin["eta2"].apply(effect_label)

reject_o, p_bonf_o, _, _ = multipletests(tbl_anova_origin["p_value"], alpha=ALPHA, method="bonferroni")
tbl_anova_origin["p_Bonferroni"] = p_bonf_o
tbl_anova_origin["Significant"] = tbl_anova_origin["p_value"] < ALPHA

print(tbl_anova_origin.round(6).to_string(index=False))
tbl_anova_origin.to_csv("04_anova_origin.csv", index=False)

Qx = tbl_anova_origin.iloc[0]["Variable"]
print(f"\n*** Qx (most significant for Origin) = {Qx} ***\n")

print(f"Tukey HSD post-hoc for Qx = {Qx} (all pairwise Origins):")
tukey_qx = pairwise_tukeyhsd(df[Qx], df["Origin"], alpha=ALPHA)
print(tukey_qx)
tukey_df = pd.DataFrame(tukey_qx.summary().data[1:], columns=tukey_qx.summary().data[0])
tukey_df.to_csv("05_tukey_Qx_origin.csv", index=False)

fig = tukey_qx.plot_simultaneous(figsize=(8, 5))
plt.title(f"Tukey HSD - {Qx} by Origin")
plt.tight_layout()
plt.savefig("05_tukey_Qx_origin.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(8, 5))
data_per_origin = [df[df["Origin"] == o][Qx].values for o in origin_order]
bp = ax.boxplot(data_per_origin, label=origin_order, patch_artist=True)
for patch, o in zip(bp['boxes'], origin_order):
    patch.set_facecolor(ORIGIN_COLORS[o]); patch.set_alpha(0.8)
for i, o in enumerate(origin_order):
    yvals = df[df["Origin"] == o][Qx].values
    ax.scatter(np.random.normal(i+1, 0.06, len(yvals)), yvals, alpha=0.25, s=10, color='black')
row0 = tbl_anova_origin.iloc[0]
ax.set_title(f"Variable: {Qx} by Origin\nOne-Way ANOVA F={row0.F_value:.2f}  "
             f"p={row0.p_value:.2e}  eta2={row0.eta2:.4f}")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig("05_boxplot_Qx_origin.png", dpi=150)
plt.close()
print("  -> Saved: 04_anova_origin.csv | 05_tukey_Qx_origin.csv/.png | 05_boxplot_Qx_origin.png\n")

# =============================================================================
# 1-F. ONE-WAY ANOVA - ROASTING
# =============================================================================
print("=" * 70)
print("1-F. ONE-WAY ANOVA: Effect of ROASTING")
print("=" * 70)

rows = []
for v in QUANT_VARS:
    F, p, eta2 = eta_squared_oneway(df, v, "Roasting")
    rows.append({"Variable": v, "F_value": F, "p_value": p, "eta2": eta2})
tbl_anova_roast = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
tbl_anova_roast["effect_size"] = tbl_anova_roast["eta2"].apply(effect_label)
reject_r, p_bonf_r, _, _ = multipletests(tbl_anova_roast["p_value"], alpha=ALPHA, method="bonferroni")
tbl_anova_roast["p_Bonferroni"] = p_bonf_r
tbl_anova_roast["Significant"] = tbl_anova_roast["p_value"] < ALPHA

print(tbl_anova_roast.round(6).to_string(index=False))
tbl_anova_roast.to_csv("06_anova_roasting.csv", index=False)

Qr = tbl_anova_roast.iloc[0]["Variable"]
print(f"\n*** Most significant for Roasting = {Qr} ***\n")

tukey_qr = pairwise_tukeyhsd(df[Qr], df["Roasting"], alpha=ALPHA)
print(f"Tukey HSD post-hoc for {Qr} by Roasting:")
print(tukey_qr)
tukey_qr_df = pd.DataFrame(tukey_qr.summary().data[1:], columns=tukey_qr.summary().data[0])
tukey_qr_df.to_csv("06_tukey_Qr_roasting.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 5))
data_per_roast = [df[df["Roasting"] == r][Qr].values for r in roasting_order]
bp = ax.boxplot(data_per_roast, label=["Light", "Medium", "Dark"], patch_artist=True)
for patch, r in zip(bp['boxes'], roasting_order):
    patch.set_facecolor(ROAST_COLORS[r]); patch.set_alpha(0.8)
row0 = tbl_anova_roast.iloc[0]
ax.set_title(f"Variable: {Qr} by Roasting\nOne-Way ANOVA F={row0.F_value:.2f}  "
             f"p={row0.p_value:.2e}  eta2={row0.eta2:.4f}")
plt.tight_layout()
plt.savefig("06_boxplot_Qr_roasting.png", dpi=150)
plt.close()
print("  -> Saved: 06_anova_roasting.csv | 06_tukey_Qr_roasting.csv | 06_boxplot_Qr_roasting.png\n")

# =============================================================================
# 1-G. TWO-WAY ANOVA - ORIGIN x ROASTING
# =============================================================================
print("=" * 70)
print("1-G. TWO-WAY ANOVA: Origin x Roasting (+ interaction)")
print("=" * 70)
print("""
MODEL : yijk = mu + alpha_i + beta_j + (alpha*beta)_ij + eps_ijk
H0_A  : no Origin effect      H0_B : no Roasting effect      H0_AB: no interaction
Practical significance: partial eta-squared per effect
  partial_eta2 = SS_effect / (SS_effect + SS_residual)
""")

rows = []
for v in QUANT_VARS:
    model = ols(f"{v} ~ C(Origin) + C(Roasting) + C(Origin):C(Roasting)", data=df).fit()
    aov = sm.stats.anova_lm(model, typ=2)
    ss_res = aov.loc["Residual", "sum_sq"]

    def p_eta2(effect):
        return aov.loc[effect, "sum_sq"] / (aov.loc[effect, "sum_sq"] + ss_res)

    rows.append({
        "Variable": v,
        "F_Origin": aov.loc["C(Origin)", "F"],
        "p_Origin": aov.loc["C(Origin)", "PR(>F)"],
        "eta2p_Origin": p_eta2("C(Origin)"),
        "F_Roasting": aov.loc["C(Roasting)", "F"],
        "p_Roasting": aov.loc["C(Roasting)", "PR(>F)"],
        "eta2p_Roasting": p_eta2("C(Roasting)"),
        "F_Interaction": aov.loc["C(Origin):C(Roasting)", "F"],
        "p_Interaction": aov.loc["C(Origin):C(Roasting)", "PR(>F)"],
        "eta2p_Interaction": p_eta2("C(Origin):C(Roasting)")
    })

tbl_2way = pd.DataFrame(rows)
print(tbl_2way.round(5).to_string(index=False))
tbl_2way.to_csv("07_anova_2way.csv", index=False)

interact = df.groupby(["Origin", "Roasting"], observed=True)[Qx].agg(["mean", "sem"]).reset_index()
fig, ax = plt.subplots(figsize=(9, 5))
for r in roasting_order:
    sub = interact[interact["Roasting"] == r]
    sub = sub.set_index("Origin").reindex(origin_order).reset_index()
    ax.errorbar(sub["Origin"], sub["mean"], yerr=sub["sem"], marker=ROAST_MARKERS[r],
                color=ROAST_COLORS[r], label={"L": "Light", "M": "Medium", "D": "Dark"}[r],
                linewidth=2, markersize=8, capsize=4)
ax.set_title(f"Interaction Plot: {Qx} ~ Origin x Roasting\n(points = group mean +/- SEM)")
ax.set_xlabel("Origin"); ax.set_ylabel(f"Mean {Qx}")
ax.legend(title="Roasting")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("07_interaction_plot.png", dpi=150)
plt.close()
print("  -> Saved: 07_anova_2way.csv | 07_interaction_plot.png\n")

# =============================================================================
# 1-H. CORRELATION ANALYSIS
# =============================================================================
print("=" * 70)
print("1-H. CORRELATION ANALYSIS (Pearson)")
print("=" * 70)

qdata = df[QUANT_VARS]
cor_mat = qdata.corr(method="pearson")
n_pairs = N_QUANT * (N_QUANT - 1) // 2
alpha_bonf_cor = ALPHA / n_pairs
print(f"Bonferroni-corrected alpha for {n_pairs} correlation pairs: {alpha_bonf_cor:.5f}\n")

p_mat = pd.DataFrame(np.ones((N_QUANT, N_QUANT)), index=QUANT_VARS, columns=QUANT_VARS)
for i, vi in enumerate(QUANT_VARS):
    for j, vj in enumerate(QUANT_VARS):
        if i != j:
            _, p = stats.pearsonr(qdata[vi], qdata[vj])
            p_mat.loc[vi, vj] = p

print("Significant correlations (p_Bonferroni < 0.05 AND |r| > 0.3):")
sig_pairs = []
for i in range(N_QUANT):
    for j in range(i + 1, N_QUANT):
        vi, vj = QUANT_VARS[i], QUANT_VARS[j]
        p_adj = p_mat.loc[vi, vj] * n_pairs
        r = cor_mat.loc[vi, vj]
        if p_adj < ALPHA and abs(r) > 0.3:
            print(f"  {vi:<12} vs {vj:<12} : r={r:+.3f}  p_uncorr={p_mat.loc[vi,vj]:.5f}  p_Bonf={p_adj:.5f}")
            sig_pairs.append((vi, vj, r, p_mat.loc[vi, vj], p_adj))

pd.DataFrame(sig_pairs, columns=["Var1", "Var2", "r", "p_uncorrected", "p_Bonferroni"]
             ).to_csv("08_significant_correlations.csv", index=False)

fig, ax = plt.subplots(figsize=(10, 9))
mask = np.triu(np.ones_like(cor_mat, dtype=bool), k=1)
sns.heatmap(cor_mat, mask=~mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.4, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title("Pearson Correlation Matrix - Coffee Aroma Compounds")
plt.tight_layout()
plt.savefig("08_correlation_plot.png", dpi=150)
plt.close()
print("  -> Saved: 08_significant_correlations.csv | 08_correlation_plot.png\n")

# =============================================================================
# 1-I. PRINCIPAL COMPONENT ANALYSIS
# =============================================================================
print("=" * 70)
print("1-I. PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("=" * 70)
print("""
Scaling rationale: variables span very different ranges (Furfural ~99,000
vs Guaiacol ~1). Without scaling to unit variance, PCA would be dominated
by high-magnitude variables regardless of biological relevance.
""")

X_scaled = StandardScaler().fit_transform(qdata)
pca = PCA(n_components=5)
scores = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_ * 100

print("Variance explained by PCs:")
for i, e in enumerate(explained, 1):
    print(f"  PC{i}: {e:.2f}%")
pd.DataFrame({"PC": [f"PC{i+1}" for i in range(5)],
              "Variance_explained_%": explained}).to_csv("09_pca_variance.csv", index=False)

pct1, pct2 = explained[0], explained[1]

fig, ax = plt.subplots(figsize=(10, 7))
for o in origin_order:
    for r in roasting_order:
        mask_or = (df["Origin"] == o) & (df["Roasting"] == r)
        if mask_or.sum() == 0:
            continue
        ax.scatter(scores[mask_or.values, 0], scores[mask_or.values, 1],
                   color=ORIGIN_COLORS[o], marker=ROAST_MARKERS[r],
                   s=45, alpha=0.8, edgecolors='white', linewidth=0.4)
origin_handles = [mpatches.Patch(color=ORIGIN_COLORS[o], label=o) for o in origin_order]
roast_handles = [plt.Line2D([0],[0], marker=ROAST_MARKERS[r], color='grey', linestyle='None',
                              markersize=8, label={"L":"Light","M":"Medium","D":"Dark"}[r])
                 for r in roasting_order]
leg1 = ax.legend(handles=origin_handles, title="Origin", loc="upper left", bbox_to_anchor=(1.01, 1))
ax.add_artist(leg1)
ax.legend(handles=roast_handles, title="Roasting", loc="lower left", bbox_to_anchor=(1.01, 0.3))
ax.axhline(0, color='grey', linestyle='--', linewidth=0.7)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.7)
ax.set_xlabel(f"PC1 ({pct1:.1f}%)")
ax.set_ylabel(f"PC2 ({pct2:.1f}%)")
ax.set_title("PCA Score Plot - Coffee Aroma Volatilomics\nColor = Origin | Marker shape = Roasting")
plt.tight_layout()
plt.savefig("09_pca_scores.png", dpi=150, bbox_inches="tight")
plt.close()

loadings = pca.components_[:2].T
fig, ax = plt.subplots(figsize=(8, 7))
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), linestyle=":", color="grey", linewidth=0.8)
for i, var in enumerate(QUANT_VARS):
    ax.annotate("", xy=(loadings[i,0], loadings[i,1]), xytext=(0,0),
                arrowprops=dict(arrowstyle="->", color="#333333", lw=1.4))
    ax.text(loadings[i,0]*1.08, loadings[i,1]*1.08, var, fontsize=9,
            color="#c0392b", fontweight="bold", ha='center', va='center')
ax.axhline(0, color='grey', linestyle='--', linewidth=0.7)
ax.axvline(0, color='grey', linestyle='--', linewidth=0.7)
ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
ax.set_aspect("equal")
ax.set_xlabel(f"PC1 ({pct1:.1f}%)")
ax.set_ylabel(f"PC2 ({pct2:.1f}%)")
ax.set_title("PCA Loading Plot - Coffee Aroma Volatilomics")
plt.tight_layout()
plt.savefig("09_pca_loadings.png", dpi=150)
plt.close()

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(range(1, 6), explained, color="#5b9bd5", edgecolor="white")
ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=9)
ax.plot(range(1, 6), explained, color="#e31a1c", marker="o", linewidth=2, markersize=6)
ax.set_xlabel("Principal Component"); ax.set_ylabel("Variance explained (%)")
ax.set_title("Scree Plot - Variance Explained by Each PC")
ax.set_ylim(0, max(explained) + 10)
plt.tight_layout()
plt.savefig("09_pca_scree.png", dpi=150)
plt.close()

contrib = pd.DataFrame(np.abs(pca.components_[:2].T), index=QUANT_VARS, columns=["PC1", "PC2"])
print("\nVariable loadings (PC1, PC2) - sorted by |PC1|:")
print(contrib.sort_values("PC1", ascending=False).round(3).to_string())
contrib.to_csv("09_pca_loadings_table.csv")
print("\n  -> Saved: 09_pca_scores.png | 09_pca_loadings.png | 09_pca_scree.png | 09_pca_variance.csv\n")

# =============================================================================
# 1-J. CLUSTER ANALYSIS
# =============================================================================
print("=" * 70)
print("1-J. CLUSTER ANALYSIS")
print("=" * 70)
print("""
A) Hierarchical clustering (Ward's linkage, Euclidean distance) on scaled data
B) K-means clustering, k chosen via silhouette score
""")
Z = linkage(X_scaled, method="ward")

sil_scores = []
for k in range(2, 11):
    km_tmp = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, km_tmp.labels_))
best_k = int(np.argmax(sil_scores)) + 2
print(f"Silhouette-optimal k = {best_k}  (score = {max(sil_scores):.3f})")

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(2, 11), sil_scores, 'o-', color='steelblue')
ax.axvline(best_k, color='red', linestyle='--')
ax.set_xlabel("Number of clusters k"); ax.set_ylabel("Average silhouette width")
ax.set_title("Silhouette Score - Choosing k for K-Means")
plt.tight_layout()
plt.savefig("10_silhouette.png", dpi=150)
plt.close()

K_CLUSTERS = best_k

# ---- Hierarchical clustering at k = K_CLUSTERS, coloured by CLUSTER ID ----
clusters_hc = fcluster(Z, t=K_CLUSTERS, criterion="maxclust")

# Distinct, neutral palette for CLUSTER IDs -- deliberately NOT the same
# colours as ORIGIN_COLORS, to avoid implying any correspondence before the
# cross-tab is examined.
NEUTRAL_PALETTE = ["#7f7f7f", "#bcbd22", "#17becf", "#8c564b",
                   "#e377c2", "#aec7e8", "#c49c94", "#9467bd"]
CLUSTER_PALETTE = NEUTRAL_PALETTE[:K_CLUSTERS]

fig, ax = plt.subplots(figsize=(20, 8))
dn_tmp = dendrogram(Z, no_labels=True, ax=ax, no_plot=True)
leaves_order = dn_tmp["leaves"]
n = len(leaves_order)

cache = {}
n_samples = X_scaled.shape[0]


def get_leaves_under(node_id, Z, n_samples, cache):
    if node_id < n_samples:
        return {node_id}
    if node_id in cache:
        return cache[node_id]
    left, right = int(Z[node_id - n_samples, 0]), int(Z[node_id - n_samples, 1])
    leaves = (get_leaves_under(left, Z, n_samples, cache)
              | get_leaves_under(right, Z, n_samples, cache))
    cache[node_id] = leaves
    return leaves


cluster_color_by_id = {c + 1: CLUSTER_PALETTE[c] for c in range(K_CLUSTERS)}


def link_color_func(node_id):
    leaves_under = get_leaves_under(node_id, Z, n_samples, cache)
    cl_ids = {clusters_hc[i] for i in leaves_under}
    if len(cl_ids) == 1:
        return cluster_color_by_id[cl_ids.pop()]
    return "#000000"


dn = dendrogram(Z, no_labels=True, ax=ax, link_color_func=link_color_func,
                above_threshold_color="#000000")

# rect.hclust-style boxes around each contiguous cluster block
cluster_per_leaf_pos = clusters_hc[leaves_order]
boundaries = []
start = 0
for i in range(1, n + 1):
    if i == n or cluster_per_leaf_pos[i] != cluster_per_leaf_pos[start]:
        boundaries.append((start, i - 1, cluster_per_leaf_pos[start]))
        start = i

ymin, ymax = ax.get_ylim()
y_label_pos = -ymax * 0.16   # space below x-axis for the summary labels
for (lo, hi, cl_id) in boundaries:
    x0 = lo * 10
    x1 = hi * 10 + 10
    rect = plt.Rectangle((x0, 0), x1 - x0, ymax * 0.985, fill=False,
                          edgecolor=cluster_color_by_id[cl_id], linewidth=1.8, zorder=10)
    ax.add_patch(rect)

    # ---- Data label below each box: n samples + Origin breakdown ----------
    mask_cl = clusters_hc == cl_id
    n_cl = mask_cl.sum()
    vals, counts = np.unique(df["Origin"].values[mask_cl], return_counts=True)
    order_idx = np.argsort(-counts)
    vals, counts = vals[order_idx], counts[order_idx]
    # Build a short multi-line composition string, e.g.:
    #   n=62
    #   Brazil:62
    breakdown_lines = [f"{v}:{c}" for v, c in zip(vals, counts)]
    label_text = f"Cluster {cl_id}  (n={n_cl})\n" + "\n".join(breakdown_lines)

    x_center = (x0 + x1) / 2
    ax.text(x_center, y_label_pos, label_text, ha="center", va="top",
            fontsize=7.5, color=cluster_color_by_id[cl_id], fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=cluster_color_by_id[cl_id], linewidth=1.0))

ax.set_ylim(y_label_pos * 1.9, ymax * 1.05)   # extend plot area to fit labels

ax.set_xticks([])
ax.set_ylabel("Height (Ward linkage distance)")
ax.set_title(f"Hierarchical Clustering Dendrogram — Coffee Aroma Volatilomics\n"
             f"Ward linkage, Euclidean distance | k = {K_CLUSTERS} "
             f"(silhouette-optimal, NOT assumed = number of Origins)\n"
             f"Colours/boxes = CLUSTER ID (unsupervised) -- compare to Origin via cross-tab below")
plt.tight_layout()
plt.savefig("10_dendrogram.png", dpi=150)
plt.close()
print("  -> Saved: 10_dendrogram.png")

# ---- Cross-tabs: cluster vs Origin, cluster vs Roasting --------------------
cross_origin = pd.crosstab(clusters_hc, df["Origin"]).reindex(columns=origin_order)
cross_roast  = pd.crosstab(clusters_hc, df["Roasting"]).reindex(columns=roasting_order)

print(f"\nHierarchical clusters (k={K_CLUSTERS}) vs Origin (counts):")
print(cross_origin.to_string())
cross_origin.to_csv("10_hc_cluster_vs_origin.csv")

cross_origin_pct = cross_origin.div(cross_origin.sum(axis=1), axis=0).mul(100).round(1)
print(f"\nHierarchical clusters (k={K_CLUSTERS}) vs Origin (% of cluster):")
print(cross_origin_pct.to_string())
cross_origin_pct.to_csv("10_hc_cluster_vs_origin_pct.csv")

print(f"\nHierarchical clusters (k={K_CLUSTERS}) vs Roasting (counts):")
print(cross_roast.to_string())
cross_roast.to_csv("10_hc_cluster_vs_roasting.csv")
print()

# ---- B) K-Means clustering, plotted on PCA space, coloured by Origin,
#         marker = K-Means cluster (analogous logic to the PCA score plot:
#         colour is the MEANINGFUL/interpretable variable; cluster ID alone
#         is an arbitrary integer with no intrinsic meaning) -----------------
km = KMeans(n_clusters=K_CLUSTERS, n_init=25, random_state=42).fit(X_scaled)
km_labels = km.labels_

cluster_markers = ["o", "s", "^", "D", "v", "P", "X", "*"][:K_CLUSTERS]

fig, ax = plt.subplots(figsize=(10, 7.5))
for o in origin_order:
    for c in range(K_CLUSTERS):
        mask = (df["Origin"].values == o) & (km_labels == c)
        if mask.sum() == 0:
            continue
        ax.scatter(scores[mask, 0], scores[mask, 1],
                   color=ORIGIN_COLORS[o], marker=cluster_markers[c],
                   s=50, alpha=0.85, edgecolors="white", linewidth=0.4)

ax.axhline(0, color="grey", linestyle="--", linewidth=0.7)
ax.axvline(0, color="grey", linestyle="--", linewidth=0.7)
ax.set_xlabel(f"PC1 ({pct1:.1f}%)")
ax.set_ylabel(f"PC2 ({pct2:.1f}%)")
ax.set_title(f"K-Means Clusters (k={K_CLUSTERS}, silhouette-optimal) on PCA Space\n"
             f"Colour = true Origin (for reference) | Marker shape = K-Means cluster ID")

leg1 = ax.legend(handles=origin_handles, title="Origin", loc="upper left",
                  bbox_to_anchor=(1.02, 1), fontsize=8, title_fontsize=9)
ax.add_artist(leg1)
marker_handles = [Line2D([0], [0], marker=cluster_markers[c], color="grey",
                          linestyle="None", markersize=8, label=f"K-Means cluster {c+1}")
                   for c in range(K_CLUSTERS)]
ax.legend(handles=marker_handles, title="K-Means cluster\n(arbitrary ID)",
          loc="lower left", bbox_to_anchor=(1.02, 0), fontsize=8, title_fontsize=9)

plt.tight_layout()
plt.savefig("10_kmeans_pca.png", dpi=150, bbox_inches="tight")
plt.close()
print("  -> Saved: 10_kmeans_pca.png")

cross_km = pd.crosstab(km_labels, df["Origin"]).reindex(columns=origin_order)
print(f"\nK-Means clusters (k={K_CLUSTERS}) vs Origin (counts):")
print(cross_km.to_string())
cross_km.to_csv("10_kmeans_cluster_vs_origin.csv")

print("\n  -> Saved: 10_silhouette.png | 10_dendrogram.png | 10_kmeans_pca.png | 10_kmeans_vs_origin.csv\n")

print("=" * 70)
print("SECTION 1 COMPLETE.")
print("=" * 70)

  SECTION 1 - DATASET ANALYSIS (Python)

Dataset: 275 rows x 15 columns
Quantitative variables (13): ['Acids', 'Alcohols', 'Aldehydes', 'Ketones', 'Esters', 'Terpenes', 'Phenols', 'Pyrazines', 'Vanillin', 'Eugenol', 'Linalool', 'Furfural', 'Guaiacol']
Q1 (first quant var) = Acids  |  Qp (last quant var) = Guaiacol
Origin1 = Brazil  |  Origin2 = Colombia

Origin counts:
Origin
Brazil          64
Vietnam         56
Peru            47
Colombia        44
India Cherry    37
Uganda          27

Roasting counts:
Roasting
M    98
D    98
L    79

1-A. VARIABLE TYPES

COLUMN         | TYPE                       | SCALE  | REASON
-----------------------------------------------------------------------
Origin         | Categorical NOMINAL         | -      | Country names, no natural order
Roasting       | Categorical ORDINAL         | -      | L < M < D (degree of roasting)
<all 13 vars>  | Quantitative CONTINUOUS     | Ratio  | Concentrations; true zero exists;
               |                   

In [5]:
# -*- coding: utf-8 -*-
"""
PTRMS_Classification — adapted from Prof. Cappellin's notebook for desktop use
Coffee Aroma Dataset — Student 06

Classifiers compared: PLS-DA, SVM, LDA-Shrinkage (PDA), Random Forest, TabPFN, XGBoost

ADAPTATION NOTES (vs. original Colab notebook):
  - Removed Google Colab-specific code (drive.mount, ipywidgets, google.sheets)
  - File path is now local; IDs/y/X columns set directly as variables
  - Fixed a bug in PLSDA_Classifier: PLSRegression.fit() requires a NUMERIC y;
    the original code passed the raw multiclass string labels directly, which
    raises "could not convert string to float". Fix: dummy-encode y (LabelBinarizer)
    before fitting PLS, exactly as standard PLS-DA does (Y = one-hot class matrix).
    The LDA step afterwards still uses the original string labels, unchanged
    from the original logic.

DATASET MAPPING (coffee aroma volatilomics):
  IDs   = Roasting (D, L, M)   <- used to split into Leave-Group-Out folds
  y     = Origin (6 classes: Brazil, Colombia, India Cherry, Peru, Uganda, Vietnam)
  X     = the 12 quantitative volatile compounds (Acids ... Guaiacol)

WHY Roasting AS "IDs" / Leave-Group-Out?
  The original notebook is designed for repeated-measurement datasets where
  IDs identify which samples belong together (e.g. same subject, same batch)
  so that grouped CV avoids leaking information between train/test. Our
  dataset has no repeated-subject structure, but using Roasting level as the
  grouping variable gives a meaningful and STRICT test:
  "Train on two roasting levels, predict Origin on the THIRD, unseen,
  roasting level." This checks whether Origin-discriminating signal in the
  volatile profile is robust across roasting conditions, not just memorised
  for one specific roast.

HOW TO RUN:
  1. pip install pandas numpy scikit-learn xgboost tabpfn
  2. (Optional, for TabPFN) See TabPFN setup instructions printed below
  3. Place STUDENT06_DATASET_coffee_aroma.csv in the same folder
  4. python ptrms_classification_coffee.py
"""

import os
os.environ["TABPFN_TOKEN"] = "tabpfn_sk_JFfi1eXBq_gi4vdcXlRzHB-qxrpfhvPQv7bOo_wj2YA"
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelBinarizer

# ─────────────────────────────────────────────────────────────────────────────
# PLS-DA classifier  (PLS for dimensionality reduction + LDA for classification)
# Same architecture as the teacher's notebook; bug-fixed for multiclass y.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.cross_decomposition import PLSRegression
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

class PLSDA_Classifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_components=2):
        """
        A classifier that combines PLS for dimensionality reduction
        followed by LDA for classification.

        Parameters
        ----------
        n_components : int
            Number of PLS components to use.
        """
        self.n_components = n_components

    def fit(self, X, y):
        """Fit the PLSDA classifier."""
        X, y = check_X_y(X, y)
        self.classes_ = np.unique(y)

        # --- FIX vs. original notebook ---------------------------------------
        # PLSRegression.fit(X, y) requires numeric y. For multiclass string
        # labels we dummy-encode y into a one-hot matrix (standard PLS-DA Y
        # block) before fitting PLS. This preserves the original intent
        # (use PLS scores as LDA input) while making it work for >2 classes.
        self._lb = LabelBinarizer()
        Y_dummy = self._lb.fit_transform(y)
        if Y_dummy.shape[1] == 1:        # binary-class edge case
            Y_dummy = np.hstack([1 - Y_dummy, Y_dummy])
        # -----------------------------------------------------------------------

        self.pls_ = PLSRegression(n_components=self.n_components)
        X_reduced = self.pls_.fit(X, Y_dummy).x_scores_

        self.lda_ = LinearDiscriminantAnalysis()
        self.lda_.fit(X_reduced, y)

        return self

    def predict(self, X):
        """Predict labels for given data."""
        check_is_fitted(self, ["pls_", "lda_"])
        X = check_array(X)
        X_reduced = self.pls_.transform(X)
        return self.lda_.predict(X_reduced)

    def predict_proba(self, X):
        """Predict class probabilities for X."""
        check_is_fitted(self, ["pls_", "lda_"])
        X = check_array(X)
        X_reduced = self.pls_.transform(X)
        return self.lda_.predict_proba(X_reduced)


# ─────────────────────────────────────────────────────────────────────────────
# Core Leave-Group-Out prediction function (UNCHANGED logic from teacher's code)
# ─────────────────────────────────────────────────────────────────────────────
def predict_class_by_label(IDs, y, X, classifier, param_grid={}, cv=5,
                            gpu=False, verbose=True):
    """
    Calculates predictions for test sets, each with a unique label in IDs,
    and selects the best parameters using GridSearchCV.

    Args:
        IDs: A pandas Series containing group labels (here: Roasting).
        y: A pandas Series or numpy array containing target labels (here: Origin).
        X: A pandas DataFrame or numpy array containing features.
        classifier: A scikit-learn classifier class.
        param_grid: A dictionary specifying parameter options for GridSearchCV.
        cv: Number of folds for cross-validation. If cv<=1, fits a single model
            directly with the given param_grid (no grid search).
        gpu: If True, move X to gpu before training (kept for compatibility;
             not used here since we run on numpy/CPU).
        verbose: If True, prints progress and metrics.

    Returns:
        predictions: Predictions for y as test sets (Leave-Group-Out).
        final_error: overall classification error (1 - accuracy).
        best_params_per_label: dict with the best parameters per group label.
    """
    unique_labels = IDs.unique()
    best_params_per_label = {}

    if not isinstance(X, np.ndarray):
        X = X.to_numpy()
    if not isinstance(y, np.ndarray):
        y = y.to_numpy()

    predictions = np.empty_like(y)

    for label in unique_labels:
        mask = (IDs == label).to_numpy() if hasattr(IDs, "to_numpy") else (IDs == label)

        X_train, X_test = X[~mask], X[mask]
        y_train, y_test = y[~mask], y[mask]

        if cv > 1:
            grid_search = GridSearchCV(
                estimator=classifier(),
                param_grid=param_grid,
                scoring='accuracy',
                cv=cv,
                verbose=verbose
            )
            grid_search.fit(X_train, y_train)
            best_model = grid_search.best_estimator_
            best_params_per_label[label] = grid_search.best_params_
        else:
            best_model = classifier(**param_grid)
            best_model.fit(X_train, y_train)
            best_params_per_label[label] = param_grid

        predictions[mask] = best_model.predict(X_test)

        if verbose:
            print(f"Group (Roasting): {label}, Best Params: {best_params_per_label[label]}")
            print(f"Test-set error for group {label}: "
                  f"{round(1.0 - accuracy_score(y_test, predictions[mask]), 4)}")

    final_error = 1.0 - accuracy_score(y, predictions)

    if verbose:
        print(f"\nOverall Classification Error: {round(final_error, 4)}")

    return predictions, final_error, best_params_per_label


# =============================================================================
# 1. CONFIGURATION — equivalent to the widget selections in the Colab notebook
# =============================================================================
print("=" * 70)
print("  COFFEE AROMA — CLASSIFIER COMPARISON (Prof. Cappellin framework)")
print("  IDs = Roasting | y = Origin | X = 13 volatile compounds")
print("=" * 70)

# --- which methods to run (set True/False as needed) -----------------------
use_rf    = True    # Random Forest
use_svm   = True    # Support Vector Machine
use_plsda = True    # PLS-DA (PLS + LDA)
use_pda   = True    # LDA with shrinkage ("PDA")
use_xgb   = True    # XGBoost
use_tab   = True    # TabPFN  (auto-skips if not installed/licensed)

# --- dataset / column configuration -----------------------------------------
# Define folder_path from the previous cell.
folder_path = '/content/drive/My Drive/Datasets'
DATA_FILE   = os.path.join(folder_path, "STUDENT06_DATASET_coffee_aroma.csv")
IDs_column  = 2   # "Roasting" is the 2nd column (1-indexed, like teacher's sliders)
y_column    = 1   # "Origin"   is the 1st column
x_first_col = 3   # quantitative compounds start at the 3rd column

# =============================================================================
# 2. LOAD DATASET
# =============================================================================
dataset = pd.read_csv(DATA_FILE)

IDs = dataset.iloc[:, IDs_column - 1]
y   = dataset.iloc[:, y_column - 1]
X   = dataset.iloc[:, x_first_col - 1:]

print(f"\nDataset shape: {dataset.shape}")
print(f"IDs (grouping variable) = '{IDs.name}'   | unique groups: {list(IDs.unique())}")
print(f"y (target)              = '{y.name}'     | classes: {list(y.unique())}")
print(f"X (features)             = {list(X.columns)}")
print(f"\nGroup sizes (Roasting):\n{IDs.value_counts().to_string()}")
print(f"\nClass sizes (Origin):\n{y.value_counts().to_string()}\n")

# =============================================================================
# 3. RUN SELECTED METHODS
# =============================================================================

# ---- PLS-DA (PLS + LDA) -----------------------------------------------------
if use_plsda:
    print("\n" + "-" * 70)
    print("Running PLS-DA ...")
    print("-" * 70)
    param_grid = {
        'n_components': [2, 3, 4, 5, 6, 7, 8]
    }
    predict_PLSDA = predict_class_by_label(
        cv=3, IDs=IDs, y=y, X=X,
        classifier=PLSDA_Classifier, param_grid=param_grid, verbose=False
    )

# ---- SVM (linear kernel, scaled features) -----------------------------------
if use_svm:
    print("\n" + "-" * 70)
    print("Running SVM ...")
    print("-" * 70)
    scaler = StandardScaler()
    param_grid = {
        'kernel': ['linear'],
        'C': [0.01, 0.1, 1.0, 10, 100, 1000, 10000, 100000],
        'degree': [3],
        'gamma': ['scale']
    }
    predict_SVM = predict_class_by_label(
        cv=3, IDs=IDs, y=y, X=scaler.fit_transform(X),
        classifier=SVC, param_grid=param_grid, verbose=False
    )

# ---- PDA (LDA with shrinkage regularisation) --------------------------------
if use_pda:
    print("\n" + "-" * 70)
    print("Running LDA-Shrinkage (PDA) ...")
    print("-" * 70)
    param_grid = {
        'solver': 'lsqr',
        'shrinkage': 'auto'
    }
    predict_PDA = predict_class_by_label(
        cv=0, IDs=IDs, y=y, X=X,
        classifier=LinearDiscriminantAnalysis, param_grid=param_grid, verbose=False
    )

# ---- Random Forest -----------------------------------------------------------
if use_rf:
    print("\n" + "-" * 70)
    print("Running Random Forest ...")
    print("-" * 70)
    param_grid = {
        'n_estimators': 1000,
        'max_features': 'sqrt'
    }
    predict_RF = predict_class_by_label(
        cv=0, IDs=IDs, y=y, X=X,
        classifier=RandomForestClassifier, param_grid=param_grid, verbose=False
    )

# ---- TabPFN -------------------------------------------------------------------
if use_tab:
    print("\n" + "-" * 70)
    print("Running TabPFN ...")
    print("-" * 70)
    try:
        from tabpfn_client import TabPFNClassifier
        param_grid = {
            'ignore_pretraining_limits': True
        }
        predict_tabPFN = predict_class_by_label(
            cv=0, IDs=IDs, y=y, X=X,
            classifier=TabPFNClassifier, param_grid=param_grid, verbose=False
        )
        tabpfn_ran = True
    except Exception as e:
        print(f"  TabPFN unavailable ({type(e).__name__}): {e}")
        use_tab = False
        tabpfn_ran = False
else:
    tabpfn_ran = False

# ---- XGBoost --------------------------------------------------------------
if use_xgb:
    print("\n" + "-" * 70)
    print("Running XGBoost  (this can take a while — grid search over many params)...")
    print("-" * 70)
    from xgboost import XGBClassifier
    from sklearn.preprocessing import LabelEncoder

    # XGBoost requires integer-encoded y
    le_xgb = LabelEncoder()
    y_xgb = pd.Series(le_xgb.fit_transform(y), index=y.index, name=y.name)

    param_grid = {
        'device': ['cpu'],
        'n_estimators': [1000],
        'eta': [0.25, 0.10, 0.05],
        'max_depth': [1, 3, 5],
        'subsample': [1, 0.8, 0.6],
        'colsample_bytree': [1, 0.75, 0.5]
    }
    predict_XGB_enc = predict_class_by_label(
        cv=3, gpu=False, IDs=IDs, y=y_xgb, X=X,
        classifier=XGBClassifier, param_grid=param_grid, verbose=False
    )
    # decode predictions back to original Origin labels for reporting
    predict_XGB = (
        le_xgb.inverse_transform(predict_XGB_enc[0]),
        predict_XGB_enc[1],
        predict_XGB_enc[2]
    )

# =============================================================================
# 4. SHOW RESULTS
# =============================================================================
print("\n" + "=" * 70)
print("RESULTS")
print("=" * 70)
print("Average Classification Error (Leave-Group-Out by Roasting)")
print("-" * 70)
if use_plsda:
    print(f"PLS-DA   : {round(predict_PLSDA[1], 4)}   (accuracy = {round(1-predict_PLSDA[1], 4)})")
if use_svm:
    print(f"SVM      : {round(predict_SVM[1], 4)}   (accuracy = {round(1-predict_SVM[1], 4)})")
if use_pda:
    print(f"PDA      : {round(predict_PDA[1], 4)}   (accuracy = {round(1-predict_PDA[1], 4)})")
if use_rf:
    print(f"RF       : {round(predict_RF[1], 4)}   (accuracy = {round(1-predict_RF[1], 4)})")
if tabpfn_ran:
    print(f"tabPFN   : {round(predict_tabPFN[1], 4)}   (accuracy = {round(1-predict_tabPFN[1], 4)})")
if use_xgb:
    print(f"XGBoost  : {round(predict_XGB[1], 4)}   (accuracy = {round(1-predict_XGB[1], 4)})")

print("\n" + "-" * 70)
print("Predictions for each sample")
print("-" * 70)

results = {'y': y}
if use_plsda:
    results['PLSDA'] = predict_PLSDA[0]
if use_svm:
    results['SVM'] = predict_SVM[0]
if use_pda:
    results['PDA'] = predict_PDA[0]
if use_rf:
    results['RF'] = predict_RF[0]
if tabpfn_ran:
    results['tabPFN'] = predict_tabPFN[0]
if use_xgb:
    results['XGBoost'] = predict_XGB[0]

df_results = pd.DataFrame(results)
print(df_results.head(15).to_string())
df_results.to_csv("classification_predictions.csv", index=False)
print(f"\n(Full predictions table saved to classification_predictions.csv, "
      f"{len(df_results)} rows)")

print("\n" + "-" * 70)
print("Confusion matrices")
print("-" * 70)

for column in df_results.columns[1:]:
    cm = pd.crosstab(df_results['y'], df_results[column],
                      rownames=['Actual'], colnames=['Predicted'])
    print(f"\nConfusion Matrix for {column}:")
    print(cm)
    cm.to_csv(f"confusion_matrix_{column}.csv")

# =============================================================================
# 5. SUMMARY BAR CHART (replaces google.colab.sheets interactive viewer)
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

methods, errors = [], []
if use_plsda: methods.append("PLS-DA"); errors.append(predict_PLSDA[1])
if use_svm:   methods.append("SVM");    errors.append(predict_SVM[1])
if use_pda:   methods.append("PDA");    errors.append(predict_PDA[1])
if use_rf:    methods.append("RF");     errors.append(predict_RF[1])
if tabpfn_ran:methods.append("TabPFN"); errors.append(predict_tabPFN[1])
if use_xgb:   methods.append("XGBoost");errors.append(predict_XGB[1])

accs = [1 - e for e in errors]
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(methods, [a*100 for a in accs],
              color=['#2980b9','#27ae60','#8e44ad','#e67e22','#c0392b','#16a085'][:len(methods)],
              edgecolor='white')
for bar, a in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f"{a*100:.1f}%", ha='center', fontweight='bold')
ax.set_ylabel("Accuracy (%)  [Leave-Group-Out by Roasting]")
ax.set_title("Classifier Comparison — Predicting Coffee Origin\n"
             "(train on 2 roasting levels, test on the 3rd, unseen, level)")
ax.set_ylim(0, 110)
ax.axhline(100/len(y.unique()), color='grey', linestyle='--',
           label=f"Random chance ({100/len(y.unique()):.1f}%)")
ax.legend()
plt.tight_layout()
plt.savefig("classifier_comparison.png", dpi=150)
plt.close()
print("\n  -> Saved: classifier_comparison.png")
print("  -> Saved: classification_predictions.csv")
print("  -> Saved: confusion_matrix_<method>.csv  (one per method)")

print("\n" + "=" * 70)
print("DONE.")
print("=" * 70)


  COFFEE AROMA — CLASSIFIER COMPARISON (Prof. Cappellin framework)
  IDs = Roasting | y = Origin | X = 13 volatile compounds

Dataset shape: (275, 15)
IDs (grouping variable) = 'Roasting'   | unique groups: ['D', 'L', 'M']
y (target)              = 'Origin'     | classes: ['Brazil', 'Colombia', 'India Cherry', 'Peru', 'Uganda', 'Vietnam']
X (features)             = ['Acids', 'Alcohols', 'Aldehydes', 'Ketones', 'Esters', 'Terpenes', 'Phenols', 'Pyrazines', 'Vanillin', 'Eugenol', 'Linalool', 'Furfural', 'Guaiacol']

Group sizes (Roasting):
Roasting
D    98
M    98
L    79

Class sizes (Origin):
Origin
Brazil          64
Vietnam         56
Peru            47
Colombia        44
India Cherry    37
Uganda          27


----------------------------------------------------------------------
Running PLS-DA ...
----------------------------------------------------------------------

----------------------------------------------------------------------
Running SVM ...
----------------------------

Found existing access token, reusing it for authentication.

00:02 Fitting... Done!
00:02 Predicting... Done!
00:02 Fitting... Done!
00:02 Predicting... Done!
00:02 Fitting... Done!
00:02 Predicting... Done!

----------------------------------------------------------------------
Running XGBoost  (this can take a while — grid search over many params)...
----------------------------------------------------------------------

RESULTS
Average Classification Error (Leave-Group-Out by Roasting)
----------------------------------------------------------------------
PLS-DA   : 0.2364   (accuracy = 0.7636)
SVM      : 0.2182   (accuracy = 0.7818)
PDA      : 0.0   (accuracy = 1.0)
RF       : 0.0909   (accuracy = 0.9091)
tabPFN   : 0.0   (accuracy = 1.0)
XGBoost  : 0.0218   (accuracy = 0.9782)

----------------------------------------------------------------------
Predictions for each sample
----------------------------------------------------------------------
         y   PLSDA     SVM     PDA      RF  tabPFN XGBoost
0   Brazil  Brazil  Brazil  Brazil  Br

In [6]:
# -*- coding: utf-8 -*-
"""
CHEMOMETRICS EXAM 2026 — STUDENT 06
SECTION 3: VALIDATION OF AN ANALYTICAL METHOD
Standard: BS EN ISO 24197:2022 — Vapour products: determination of
          e-liquid vaporised mass (EVM) and aerosol collected mass (ACM)

This script:
  1. Reproduces the precision data (r, R, sr, sR, sL) reported in the
     standard (Table 1 - Study 1, Table 2 - Study 2)
  2. Calculates repeatability/reproducibility SDs from the published limits
  3. Prints a full worked example
  4. Prints the validation plan a new laboratory should follow
     (precision: repeatability + intermediate precision; trueness: bias)

HOW TO RUN:
  pip install pandas numpy matplotlib
  python section3_method_validation.py
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("=" * 70)
print("  SECTION 3 - BS EN ISO 24197:2022 - METHOD VALIDATION")
print("=" * 70)

print("""
WHAT IS BS EN ISO 24197:2022?
  Gravimetric determination of:
    - ACM = Aerosol Collected Mass  (deposited on glass fibre filter)
            Formula: m_ACM = m_f - m_i   [filter mass after - before vaping, mg]
    - EVM = E-liquid Vaporised Mass (lost from the vaping device)
            Formula: m_EVM = m_i - m_f   [device mass before - after vaping, mg]

  Key apparatus:
    - Routine analytical vaping machine (ISO 20768)
    - Analytical balance: >= 1 mg precision, 0.1 mg display
    - Glass fibre filter: >= 99.9% retention of particles >= 0.3 micrometers

KEY RELATIONSHIPS (ISO 5725 framework) - MUST KNOW FOR THE EXAM:
  r = 2.8 x sr     ->   sr = r / 2.8      (repeatability SD from repeatability limit)
  R = 2.8 x sR     ->   sR = R / 2.8      (reproducibility SD from reproducibility limit)
  sR^2 = sr^2 + sL^2                       (between-lab variance component)
       ->  sL = sqrt(sR^2 - sr^2)

  ORIGIN OF THE FACTOR 2.8:
    factor = 2 x sqrt(2) x 1.96  ~=  2.77  ~=  2.8
      2     = comparing two independent measurements
      sqrt(2) = propagation of two independent standard errors
      1.96  = z(0.025), the z-value for a 95% two-sided confidence interval

  MEANING:
    If |x1 - x2| <= r  -> not significantly different (SAME lab, same conditions)
    If |x1 - x2| <= R  -> not significantly different (DIFFERENT labs)
    Note: sR >= sr ALWAYS, because reproducibility includes between-lab variability
          in addition to repeatability (within-lab) variability.
""")

# =============================================================================
# TABLE 1 — Study 1 (2015), 80 puffs, 18 laboratories
# =============================================================================
print("=" * 70)
print("TABLE 1 (Study 1, 2015, 80 puffs, 18 laboratories)")
print("=" * 70)

study1 = pd.DataFrame({
    "Sample": ["A", "B", "C", "D", "A", "B", "C", "D"],
    "Analyte": ["mACM"] * 4 + ["mEVM"] * 4,
    "Composition": [
        "0% Nic, Gly:PG=70:30",
        "2.4% Nic, Gly:PG=70:30",
        "5.4% Nic, Gly:PG=70:30",
        "2.4% Nic, Gly:PG=100:0"
    ] * 2,
    "Mean_mg": [148, 127, 109, 142, 137, 120, 105, 136],
    "r_mg":    [42,  43,  43,  48,  40,  42,  43,  48],
    "R_mg":    [69,  72,  51,  58,  74,  72,  51,  55],
    "r_pct":   [28.60, 33.50, 39.40, 33.90, 28.97, 34.80, 40.71, 34.99],
    "R_pct":   [46.60, 56.70, 47.20, 40.50, 53.88, 60.07, 48.49, 40.19],
})

study1["sr_mg"] = (study1["r_mg"] / 2.8).round(2)
study1["sR_mg"] = (study1["R_mg"] / 2.8).round(2)
study1["sL_mg"] = np.sqrt(np.clip(study1["sR_mg"]**2 - study1["sr_mg"]**2, 0, None)).round(2)

print(study1.to_string(index=False))
study1.to_csv("13_iso24197_study1.csv", index=False)

# =============================================================================
# TABLE 2 — Study 2 (2019), 75 puffs, 11 laboratories
# =============================================================================
print("\n" + "=" * 70)
print("TABLE 2 (Study 2, 2019, 75 puffs, 11 laboratories)")
print("Aspire Nautilus tank, 1.8 ohm coil, Evolv power unit")
print("=" * 70)

study2 = pd.DataFrame({
    "Sample": ["A", "B", "C", "A", "B", "C"],
    "Analyte": ["mACM"] * 3 + ["mEVM"] * 3,
    "Flavour": ["Unflavoured", "Tobacco", "Tobacco/Menthol"] * 2,
    "Mean_mg": [731, 705, 691, 719, 696, 683],
    "r_mg":    [67,  77,  118, 62,  82,  122],
    "R_mg":    [204, 275, 264, 190, 269, 254],
    "r_pct":   [9.20, 10.90, 17.10, 8.60, 11.80, 17.90],
    "R_pct":   [27.80, 39.00, 38.20, 26.40, 38.60, 37.20],
})

study2["sr_mg"] = (study2["r_mg"] / 2.8).round(2)
study2["sR_mg"] = (study2["R_mg"] / 2.8).round(2)
study2["sL_mg"] = np.sqrt(np.clip(study2["sR_mg"]**2 - study2["sr_mg"]**2, 0, None)).round(2)

print(study2.to_string(index=False))
study2.to_csv("13_iso24197_study2.csv", index=False)

# =============================================================================
# WORKED EXAMPLE
# =============================================================================
print("\n" + "=" * 70)
print("WORKED EXAMPLE: Study 1, Sample A, mACM")
print("=" * 70)

ex_mean, ex_r, ex_R = 148, 42, 69
ex_sr = ex_r / 2.8
ex_sR = ex_R / 2.8
ex_sL = np.sqrt(ex_sR**2 - ex_sr**2)
cv_r  = 100 * ex_sr / ex_mean
cv_R  = 100 * ex_sR / ex_mean

print(f"""
From Table 1:
  Mean = {ex_mean} mg  |  r = {ex_r} mg  |  R = {ex_R} mg

STEP 1 - Repeatability SD:
  sr = r / 2.8 = {ex_r} / 2.8 = {ex_sr:.2f} mg
  CV_r = (sr / Mean) x 100 = ({ex_sr:.2f} / {ex_mean}) x 100 = {cv_r:.2f}%

STEP 2 - Reproducibility SD:
  sR = R / 2.8 = {ex_R} / 2.8 = {ex_sR:.2f} mg
  CV_R = (sR / Mean) x 100 = {cv_R:.2f}%

STEP 3 - Between-lab SD:
  sL = sqrt(sR^2 - sr^2) = sqrt({ex_sR:.2f}^2 - {ex_sr:.2f}^2)
     = sqrt({ex_sR**2:.2f} - {ex_sr**2:.2f}) = sqrt({ex_sR**2-ex_sr**2:.2f}) = {ex_sL:.2f} mg

INTERPRETATION:
  - Two measurements from the SAME lab:
    If |x1 - x2| <= {ex_r} mg -> not significantly different (95% confidence)
  - Two measurements from DIFFERENT labs:
    If |x1 - x2| <= {ex_R} mg -> not significantly different (95% confidence)
  - The between-lab component (sL = {ex_sL:.2f} mg) is LARGER than the
    within-lab component (sr = {ex_sr:.2f} mg)
    -> inter-laboratory variability is the dominant source of uncertainty
""")

# Visualisation: r vs R vs sr, sR, sL across both studies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

combined = pd.concat([
    study1.assign(Study="Study 1 (80 puffs)"),
    study2.assign(Study="Study 2 (75 puffs)")
], ignore_index=True)

ax = axes[0]
x = np.arange(len(combined))
width = 0.35
ax.bar(x - width/2, combined["sr_mg"], width, label="sr (repeatability)", color="#3498db")
ax.bar(x + width/2, combined["sR_mg"], width, label="sR (reproducibility)", color="#e74c3c")
labels = combined["Study"].str[:7] + "-" + combined["Sample"] + "-" + combined["Analyte"]
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=75, fontsize=7)
ax.set_ylabel("Standard deviation (mg)")
ax.set_title("Repeatability vs Reproducibility SD\n(ISO 24197, all samples)")
ax.legend()

ax = axes[1]
ax.scatter(combined["sr_mg"], combined["sL_mg"], s=80, c=combined["Mean_mg"],
           cmap="viridis", edgecolors="black")
ax.plot([0, combined["sr_mg"].max()*1.2], [0, combined["sr_mg"].max()*1.2],
        linestyle="--", color="grey", label="sL = sr (equal contribution)")
ax.set_xlabel("sr (within-lab SD, mg)")
ax.set_ylabel("sL (between-lab SD, mg)")
ax.set_title("Within-lab vs Between-lab Variability\n(colour = mean mass, mg)")
cbar = plt.colorbar(ax.collections[0], ax=ax)
cbar.set_label("Mean mass (mg)")
ax.legend()

plt.tight_layout()
plt.savefig("13_precision_comparison.png", dpi=150)
plt.close()
print("  -> Saved: 13_iso24197_study1.csv | 13_iso24197_study2.csv | 13_precision_comparison.png\n")

# =============================================================================
# VALIDATION PLAN FOR A NEW LABORATORY
# =============================================================================
print("=" * 70)
print("VALIDATION PLAN FOR A NEW LABORATORY")
print("=" * 70)
print("""
PARAMETERS TO VALIDATE (for a standardised gravimetric method):
  1. PRECISION
     a) Repeatability        (within-lab, same-day precision)
     b) Intermediate precision (within-lab, different-day precision)
     [c) Reproducibility: only assessable via inter-laboratory comparison]
  2. TRUENESS (Bias / Recovery)

-----------------------------------------------------------
1a. REPEATABILITY VALIDATION
-----------------------------------------------------------
Experiment:
  - Prepare the SAME sample (same e-liquid, same vaping device/settings)
  - Perform n >= 10 replicate measurements
    (same analyst, same day, same instrument -> "repeatability conditions")
  - Record: x1, x2, ..., xn

Calculations:
  x_bar          = (1/n) * sum(xi)                         [mean]
  sr_lab         = sqrt( sum((xi - x_bar)^2) / (n-1) )      [repeatability SD, your lab]
  r_lab          = 2.8 * sr_lab                              [repeatability limit, your lab]
  CV(%)          = 100 * sr_lab / x_bar

Decision:
  If r_lab <= r_standard (e.g. 42 mg, Table 1 Sample A) -> repeatability ACCEPTABLE
  If r_lab >  r_standard                                -> investigate variability sources
    (balance calibration, leak checks, puff-volume checks, operator technique)

-----------------------------------------------------------
1b. INTERMEDIATE PRECISION
-----------------------------------------------------------
Experiment:
  - Same sample, >= 3 different days, same analyst (or different analysts)
  - n_i replicates per day

Calculations:
  Grand mean        : x_bar_grand = mean of ALL measurements (all days pooled)
  Within-day SD      : s_within  = pooled sr from each day's replicates
  Between-day SD      : s_day = sqrt( max(0, s_intermediate^2 - s_within^2 / n_per_day) )
  Intermediate precision SD : s_I = sqrt( s_within^2 + s_day^2 )

  (Equivalently: a one-way ANOVA with "Day" as factor gives the within-day
   and between-day variance components directly via the mean squares.)

-----------------------------------------------------------
2. TRUENESS (BIAS) VALIDATION
-----------------------------------------------------------
Experiment:
  Option A: use a Certified Reference Material (CRM) with a certified
            ACM/EVM value
  Option B: recovery study -- prepare spiked samples with a known added mass
  Measure n >= 6 replicates -> obtain x_bar_measured

Calculations:
  bias            = x_bar_measured - x_reference
  relative_bias(%) = 100 * bias / x_reference

Hypothesis test (one-sample t-test):
  H0: bias = 0   (no systematic error)
  H1: bias != 0
  t  = bias / (sr_lab / sqrt(n)),   df = n - 1
  If |t| > t_critical(alpha/2, df) -> significant bias -> method needs correction
  (e.g. recalibrate balance, correct for filter-pad moisture, re-train operator)

-----------------------------------------------------------
SUMMARY TABLE - VALIDATION PARAMETERS
-----------------------------------------------------------
Parameter            | Experiment             | Reference value (Study 1, Sample A)
---------------------|-------------------------|--------------------------------------
Repeatability (r)    | n>=10 replicates/day    | r <= 42 mg
Repeatability SD (sr)| n>=10 replicates/day    | sr <= 15.00 mg
Reproducibility SD(sR)| Inter-lab study         | sR = 24.64 mg (reference; cannot be
                      |                          | validated by a single lab alone)
Between-lab SD (sL)  | Inter-lab study         | sL = 19.54 mg
Bias                 | CRM or spike-recovery   | bias not significantly != 0 (t-test)
""")

# =============================================================================
# DONE
# =============================================================================
print("=" * 70)
print("SECTION 3 COMPLETE.")
print("  -> CSV/PNG files saved with prefix '13_'")
print("=" * 70)

  SECTION 3 - BS EN ISO 24197:2022 - METHOD VALIDATION

WHAT IS BS EN ISO 24197:2022?
  Gravimetric determination of:
    - ACM = Aerosol Collected Mass  (deposited on glass fibre filter)
            Formula: m_ACM = m_f - m_i   [filter mass after - before vaping, mg]
    - EVM = E-liquid Vaporised Mass (lost from the vaping device)
            Formula: m_EVM = m_i - m_f   [device mass before - after vaping, mg]

  Key apparatus:
    - Routine analytical vaping machine (ISO 20768)
    - Analytical balance: >= 1 mg precision, 0.1 mg display
    - Glass fibre filter: >= 99.9% retention of particles >= 0.3 micrometers

KEY RELATIONSHIPS (ISO 5725 framework) - MUST KNOW FOR THE EXAM:
  r = 2.8 x sr     ->   sr = r / 2.8      (repeatability SD from repeatability limit)
  R = 2.8 x sR     ->   sR = R / 2.8      (reproducibility SD from reproducibility limit)
  sR^2 = sr^2 + sL^2                       (between-lab variance component)
       ->  sL = sqrt(sR^2 - sr^2)

  ORIGIN OF THE FACTOR 2